# 02 — Preparación de datos (Local)

Versión local del notebook de preparación de datos. No requiere Colab ni GCS.

Este notebook:
1. Configura paths locales usando `setup_local()`
2. Extrae los tarballs de imágenes pre-descargadas (`data/processed/*.tar`)
3. Descarga imágenes faltantes desde internet (idempotente)
4. Genera splits estratificados train/val/test

**Requiere que ya existan localmente:**
- `data/preprocessed/pants_1.csv`
- `data/preprocessed/tops_1.csv`
- `data/processed/images_pants.tar` (opcional, acelera el proceso)
- `data/processed/images_tops.tar` (opcional, acelera el proceso)

## 1. Setup local

In [ ]:
import sys
from pathlib import Path

# Detectar raíz del repo (subimos desde notebooks/)
REPO_ROOT = Path('.').resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.colab import setup_local
config = setup_local(repo_root=REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('CSVs:')
print(f"  pants: {config['data']['pants_csv']}")
print(f"  tops:  {config['data']['tops_csv']}")
print('Paths:')
for k, v in config['paths'].items():
    print(f'  {k}: {v}')

## 2. Instalar dependencias (si hace falta)

In [ ]:
# Descomentar si necesitás instalar dependencias:
# !pip install -q -r {REPO_ROOT / 'requirements.txt'}

## 3. Extraer tarballs de imágenes (cache local)

Si ya corriste el notebook 02 de Colab antes y tenés los tarballs en `data/processed/`,
los extraemos para no re-descargar las imágenes desde internet.

In [ ]:
import subprocess
import os

def extract_tarball(ds: str) -> bool:
    """Extrae el tarball local de imágenes si existe."""
    # Soportamos .tar y .tar.gz
    tar_path = REPO_ROOT / f'data/processed/images_{ds}.tar'
    tar_gz_path = REPO_ROOT / f'data/processed/images_{ds}.tar.gz'

    if tar_gz_path.exists():
        src = tar_gz_path
        tar_flags = 'xzf'
    elif tar_path.exists():
        src = tar_path
        tar_flags = 'xf'
    else:
        print(f'No existe tarball para {ds}, se descargará desde internet.')
        return False

    dest = config['paths'][f'images_{ds}']
    os.makedirs(dest, exist_ok=True)

    # Verificar si ya hay imágenes extraídas
    existing = list(Path(dest).glob('*.jpg'))
    if len(existing) > 100:
        print(f'✓ Cache {ds} ya extraído ({len(existing)} imágenes). Salteando.')
        return True

    print(f'Extrayendo {src.name} → {dest} ...')
    result = subprocess.run(
        ['tar', tar_flags, str(src), '-C', str(dest)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'Error extrayendo {src.name}: {result.stderr}')
        return False

    n_imgs = len(list(Path(dest).glob('*.jpg')))
    print(f'✓ Cache {ds} extraído ({n_imgs} imágenes).')
    return True

pants_cache_ok = extract_tarball('pants')
tops_cache_ok  = extract_tarball('tops')

## 4. Descargar imágenes faltantes

Descarga imágenes que no estén en el cache local.
Si el tarball se extrajo arriba, esta celda es muy rápida (solo verifica el cache).

In [ ]:
SAMPLE_PANTS = 25000
SAMPLE_TOPS  = 25000

from src.data.downloader import download_csv_images

# Descarga de Pants
log_pants = download_csv_images(
    csv_path=config['data']['pants_csv'],
    cache_dir=config['paths']['images_pants'],
    sample=SAMPLE_PANTS,
    workers=8,
    timeout=15,
    log_path=f"{config['paths']['splits']}/pants_download_log.csv",
)
print(f"\nPants OK: {log_pants['success'].sum()}/{len(log_pants)}")

# Descarga de Tops
log_tops = download_csv_images(
    csv_path=config['data']['tops_csv'],
    cache_dir=config['paths']['images_tops'],
    sample=SAMPLE_TOPS,
    workers=8,
    timeout=15,
    log_path=f"{config['paths']['splits']}/tops_download_log.csv",
)
print(f"\nTops OK: {log_tops['success'].sum()}/{len(log_tops)}")

## 5. Splits estratificados (filtrando por imágenes descargadas)

In [ ]:
from src.data.splits import generate_splits

# Pants
pants_urls = set(log_pants[log_pants['success']]['url'])
pants_paths = generate_splits(
    csv_path=config['data']['pants_csv'],
    output_dir=config['paths']['splits'],
    stratify_col=config['data']['splits']['stratify_col'],
    train_ratio=config['data']['splits']['train_ratio'],
    val_ratio=config['data']['splits']['val_ratio'],
    test_ratio=config['data']['splits']['test_ratio'],
    min_samples_per_class=config['data']['splits']['min_samples_per_class'],
    seed=config['seed'],
    filter_urls=pants_urls,
)
print('Splits pants:', pants_paths)

In [ ]:
# Tops
tops_urls = set(log_tops[log_tops['success']]['url'])
tops_paths = generate_splits(
    csv_path=config['data']['tops_csv'],
    output_dir=config['paths']['splits'],
    stratify_col=config['data']['splits']['stratify_col'],
    train_ratio=config['data']['splits']['train_ratio'],
    val_ratio=config['data']['splits']['val_ratio'],
    test_ratio=config['data']['splits']['test_ratio'],
    min_samples_per_class=config['data']['splits']['min_samples_per_class'],
    seed=config['seed'],
    filter_urls=tops_urls,
)
print('Splits tops:', tops_paths)

## 6. Sanity check final

In [ ]:
import pandas as pd

SPLITS_DIR = config['paths']['splits']
print('=== Resumen ===')
for ds in ['pants_1', 'tops_1']:
    for split in ['train', 'val', 'test']:
        p = f"{SPLITS_DIR}/{ds}_{split}.csv"
        try:
            df = pd.read_csv(p)
            print(f'  {ds}/{split}: {len(df):,} filas')
        except Exception as e:
            print(f'  {ds}/{split}: ERROR — {e}')

print('\n✓ Listo para entrenar. Ejecuta 03_train_pants_local.ipynb')